In [1]:
import numpy as np 
from vosk import KaldiRecognizer , Model
import wave 
import librosa 
import soundfile as sf 
from IPython.display import Audio , display
from scipy.io.wavfile import write
import json

In [2]:
from faster_whisper import WhisperModel

/home/waad/Documents/my_ml_project/myenv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import unicodedata 

def remove_accents(text):
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

In [4]:
import re
import contractions
from num2words import num2words

In [5]:
## text preprocessing

def text_preprocessing(text):
    text = text.lower().strip()
    text = contractions.fix(text)
    text = remove_accents(text)
    def convert_decimal(match):
        number = match.group(0)
        integer, decimal = number.split(".")
        integer_words = num2words(int(integer))
        decimal_words = num2words(int(decimal))
        return f"{integer_words} point {decimal_words}"
        
    text = re.sub(r"\b\d+\.\d+\b", convert_decimal, text)
    text = re.sub(r"\b\d+\b", lambda x: num2words(int(x.group(0))), text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.split()

In [6]:
import difflib
from itertools import zip_longest

In [7]:
## sequance matching 

def sequance_matching_score(target_tokens, vosk_tokens):

    matcher = difflib.SequenceMatcher(None, target_tokens, vosk_tokens)
    
    report = []
    correct_words_count = 0
    total_teacher_words = len(target_tokens)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        
        if tag == 'equal':
            chunk_len = i2 - i1
            correct_words_count += chunk_len
            
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "match"
                })

        elif tag == 'delete':
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "Missed"
                })

        elif tag == 'replace':

            teacher_chunk = target_tokens[i1:i2]
            vosk_chunk = vosk_tokens[j1:j2]
            
            for t_word, v_word in zip_longest(teacher_chunk, vosk_chunk, fillvalue=None):
                
                if t_word is None:
                    break  

                if v_word is None:
                    report.append({
                        "word": t_word,
                        "status": "Missed"
                    })
                    continue 

                else:
                    similarity = difflib.SequenceMatcher(None, t_word, v_word).ratio()
                    
                    if similarity >= 0.7:
                        correct_words_count += 1
                        report.append({
                            "word": t_word, 
                            "status": f"Accepted typo ({int(similarity*100)}%)"
                        })
                    else:
                        report.append({
                            "word": t_word, 
                            "status": f"Wrong word. Heard '{v_word}'"
                        })

    if total_teacher_words == 0:
        final_score = 0
    else:
        final_score = int((correct_words_count / total_teacher_words) * 100)

    return final_score, report


In [8]:
def check_fuzzy_keywords(child_speech, keywords, threshold=0.7):
    missing_words = []
    for target in keywords:
        found = False

  
        if target in child_speech:
            found = True
        else:
            for word in child_speech:
                similarity = difflib.SequenceMatcher(None, target, word).ratio()
                if similarity >= threshold:
                    found = True
                    break
        
        if not found:
            missing_words.append(target)

    if len(missing_words) == 0:
        return True
    else:
        return False
    


In [9]:
small = WhisperModel("small.en", device="cpu", compute_type="int8")

In [10]:
from glob import glob
test_data = glob('final_test_audios/audio*.wav')


In [11]:
base = WhisperModel("base.en", device="cpu", compute_type="int8")

In [12]:
model_path = "vosk-model-small-en-us-0.15"
vosk_small = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-small-en-us-0.15/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from vosk-model-small-en-us-0.15/graph/HCLr.fst vosk-model-small-en-us-0.15/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-small-en-us-0.15/graph/phones/word_boundary.int


In [13]:
model_path = "vosk-model-en-us-0.22"
vosk_us = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:11:12:13:14:15
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-en-us-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from vosk-model-en-us-0.22/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:297) Loading words from vosk-model-en-us-0.22/graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-en-us-0.22/graph/phones/word_boundary.int
LOG (VoskAPI:ReadDataFiles():model.cc:315) Loading subtract 

In [14]:
tiny = WhisperModel("tiny.en", device="cpu", compute_type="int8")

In [15]:
from glob import glob
test_data = glob('final_test_audios/audio*.wav')

In [16]:
def audio_test_vosk(audio_path,recognizer):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()
    recognizer.AcceptWaveform(raw_noisy_audio)
    final_result = json.loads(recognizer.FinalResult())
    full_transcript = final_result.get("text","")
    return full_transcript
    

In [17]:
def audio_test_whisper(file_path, model):
        segments, info = model.transcribe(file_path, beam_size=5)
        text = " ".join([segment.text for segment in segments])
        return text 

In [18]:
us_recognizer = KaldiRecognizer(vosk_us,16000)
us_recognizer.SetWords(True)

In [19]:
small_recognizer = KaldiRecognizer(vosk_small,16000)
small_recognizer.SetWords(True)

In [20]:

j=0
for i in test_data:
    print(f"{j}:")
    print(audio_test(i,us_recognizer))
    j+= 1
    print("\n")

0:


NameError: name 'audio_test' is not defined

In [ ]:

j=0
for i in test_data:
    print(f"{j}:")
    print(audio_test(i,small_recognizer))
    j+= 1
    print("\n")

In [21]:
targets = [
    "august",
    "may",
    "July",
    "November",
    "February",
    "Have three pencils.",
    "I love my friends.",
    "The baby is sleeping",
    "She is reading a story",
    "I love my father.",
    "It is a square.",
    "It is a circle. ...",
    "I love Strawberry. ...",
    "six years old.",
    "I don't like broccoli",
    "nice to meet you!",
    "This is a circle.",
    "I like pizza.",
    "It is a square.",
    "I like the color blue.",
    "1, 2, 3, 4",
    "The dog is running",
    "The sky is blue",
    "The sun is yellow.",
    "The ball is red.",
    "Red car",
    "one plus one is two",
    "A square.",
    "One, two, three, four, five.",
    "It's the dog!",
    "This is my mom.",
    "I love my mom.",
    "Good morning, good night.",
    "Strawberry are red.",
    "Sunday, Monday, Tuesday, Wednesday, Thursday, Friday",
    "The cat is under the",
    "I love my family.",
    "It is sunny today.",
    "Nice to meet you.",
    "There are 4 cats",
    "She is sitting down.",
    "I am fine, thank you.",
    "It is a square.",
    "1, 2, 3, 4, 5",
    "I am 6 years old",
    "The cat is under the chair.",
    "I like the color blue.",
    "I love pizza!",
    "He is reading his",
    "She is reading a story.",
    "the is sleeping now",
    "3 pencils ....",
    "I have one bag.",
    "I have one",
    "This is my mom",
    "Thank my mom",
    "There are two bags.",
    "Their are welcome.",
    "Have a nice day!",
    "Thank you very much.",
    "my dad",
    "There are two dogs.",
    "You are welcome.",
    "He is jumping high.",
    "I am  playing now!",
    "He is reading a book.",
    "Hello, how are you?",
    "I am Eating an apple.",
    "I am drinking water",
    "The rabbit is small.",
    "The sun is yellow.",
    "The book is on the table.",
    "The dog is running",
    "Apple is red.",
    "The cow is big!",
    "The fish is in the water.",
    "The shoes are in the box.",
    "The boy is at school.",
    "The bird is fly",
    "The is green",
    "The sky is blue.",
    "The ball is red.",
    "Mangoes are orange.",
    "They are 4 cats.",
    "Grass green",
    "I am fine, thank you!",
    "Have a nice day!",
    "Reading a book.",
    "Pink Jacket",
    "He touch grass.",
    "The cat is on the tree.",
    "It's sunny today.",
    "I have one bag.",
    "Hello, how are you?",
    "The ball is red.",
    "Seven days a week.",
    "I don't like broccoli.",
    "I am six.",
    "I like red.",
    "A square.",
    "Four apples.",
    "In the forest.",
    "The cow is big.",
    "1, 2, 3, 4, 5",
    "I like the color blue",
    "Monday, Tuesday, Wednesday, Thursday",
    "I have three pencils",
    "There are two dogs",
    "This is my first time.",
    "I can count to 10",
    "my family is nice",
    "I have a brother",
    "I have a sister.",
    "See you later!",
    "this is my dad",
    "This is my mom.",
    "Hello, how are you?",
    "Thank you very much.",
    "I am playing now",
    "I am drinking water",
    "Have a nice day!",
    "The boy is at school.",
    "The Apple is red.",
    "I am eating an apple",
    "The dog is running",
    "The book is on the table.",
    "The girl is at home.",
    "The fish is in the water.",
    "the toy is next to the bed",
    "the shoes are in the box",
    "it is sunny today",
    "The bird can fly.",
    "The grass is green",
    "The sky is blue.",
    "The sun is yellow.",
    "The cow is big!",
    "The rabbit is small.",
    "The cat is black.",
    "Nice to meet you!",
    "I am fine. Thank you",
    "I like strawberries",
    "There are four cats.",
    "I love pizza.",
    "I am 6 years old.",
    "Good morning. Good night.",
    "mangoes are orange",
    "It's a square.",
    "He is brushing his hair.",
    "She is sitting down.",
    "This is a circle.",
    "The cat is under the chair."
]

In [22]:
import time 

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_whisper(i, tiny)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_whisper(i, base)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_whisper(i, small)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    # Test Vosk_us
    start = time.time()
    raw_vosk_us = audio_test_vosk(i, us_recognizer)
    dur_vosk_us = time.time() - start
    vosk_us_latency.append(dur_vosk_us)
    vosk_us_norm = text_preprocessing(raw_vosk_us)

    # Test Vosk_small
    start = time.time()
    raw_vosk_small = audio_test_vosk(i, small_recognizer)
    dur_vosk_small = time.time() - start
    vosk_small_latency.append(dur_vosk_small)
    vosk_small_norm = text_preprocessing(raw_vosk_small)
    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print(f"Vosk us:  {vosk_us_norm}  (Time: {dur_vosk_us:.3f}s)")
    print(f"Vosk Small: {vosk_small_norm}  (Time: {dur_vosk_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    score_vosk_us, report_vosk_us = sequance_matching_score(target_norm, vosk_us_norm)
    score_vosk_small, report_vosk_small = sequance_matching_score(target_norm, vosk_small_norm)

    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)
    vosk_us_acc_shadwing.append(score_vosk_us)
    vosk_small_acc_shadwing.append(score_vosk_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")

    print("--- VOSK US REPORT ---")
    for item in report_vosk_us:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")

    print("--- VOSK SMALL REPORT ---")
    for item in report_vosk_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} | Vosk us: {score_vosk_us}  | Vosk Small: {score_vosk_small}")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]
shadow_Vosk_us_pass = [score >= threshold for score in vosk_us_acc_shadwing]
shadow_Vosk_small_pass = [score >= threshold for score in vosk_small_acc_shadwing]

shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100
shadow_Vosk_us_acc = sum(shadow_Vosk_us_pass) / len(shadow_Vosk_us_pass) * 100
shadow_Vosk_small_acc = sum(shadow_Vosk_small_pass) / len(shadow_Vosk_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)
shadow_Vosk_us_avg = sum(vosk_us_acc_shadwing) / len(vosk_us_acc_shadwing)
shadow_Vosk_small_avg = sum(vosk_small_acc_shadwing) / len(vosk_small_acc_shadwing)
# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)
avg_time_vosk_us = sum(vosk_us_latency) / len(vosk_us_latency)
avg_time_vosk_small = sum(vosk_small_latency) / len(vosk_small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")
print(f"Shadowing Accuracy Vosk US:    {shadow_Vosk_us_acc:.2f}%")
print(f"Shadowing Accuracy Vosk Small: {shadow_Vosk_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")
print(f"Shadowing Avg Score - Vosk US:    {shadow_Vosk_us_avg:.2f}")
print(f"Shadowing Avg Score - Vosk Small: {shadow_Vosk_small_avg:.2f}")
print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")
print(f"Avg Latency (sec) - Vosk US:    {avg_time_vosk_us:.4f}s")
print(f"Avg Latency (sec) - Vosk Small: {avg_time_vosk_small:.4f}s")

Target:        ['august']
Whisper Tiny:  ['oh', 'ben']  (Time: 1.740s)
Whisper Base:  ['oh', 'guys']  (Time: 1.120s)
Whisper Small: ['oh', 'good']  (Time: 11.144s)
Vosk us:  ['all', 'good']  (Time: 3.986s)
Vosk Small: ['oh', 'good']  (Time: 0.452s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- VOSK US REPORT ---
august          | Wrong word. Heard 'all'
_____________________________________
--- VOSK SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 | Vosk us: 0  | Vosk Small: 0



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 1.716s)
Whisper Base:  ['may']  (Time: 1.097s)
Whi

In [26]:
def audio_test_denoised_whisper(file_path, model, beam_size=5, condition=False):
        segments, info = model.transcribe(file_path,
                                          language="en",
                                          vad_filter=True,
                                          beam_size=beam_size,
                                          condition_on_previous_text=condition)
        text = " ".join([segment.text for segment in segments])
        return text 

In [25]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_denoised_whisper(i, tiny)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_denoised_whisper(i, base)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_denoised_whisper(i, small)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    
    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} ")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]


shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")

print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")


Target:        ['august']
Whisper Tiny:  ['oh', 'very', 'good']  (Time: 2.823s)
Whisper Base:  ['oh', 'baby']  (Time: 1.621s)
Whisper Small: ['oh', 'good']  (Time: 12.380s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 2.308s)
Whisper Base:  ['may']  (Time: 1.407s)
Whisper Small: ['me']  (Time: 4.222s)



--- TINY REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
--- BASE REPORT ---
may             | match
_____________________________________
--- SMALL REPORT ---
may             | Wrong word. Heard 'me'
________________________________

In [27]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_denoised_whisper(i, tiny,True)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_denoised_whisper(i, base,True)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_denoised_whisper(i, small,True)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    
    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} ")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]


shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")

print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")


Target:        ['august']
Whisper Tiny:  ['oh', 'ben']  (Time: 2.071s)
Whisper Base:  ['oh', 'baby']  (Time: 1.885s)
Whisper Small: ['oh', 'good']  (Time: 11.772s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 1.753s)
Whisper Base:  ['may']  (Time: 1.924s)
Whisper Small: ['me']  (Time: 4.107s)



--- TINY REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
--- BASE REPORT ---
may             | match
_____________________________________
--- SMALL REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
Sco

In [28]:

tiny_acc_shadwing = []
tiny_latency = [] 

base_acc_shadwing = []
base_latency = [] 

small_acc_shadwing = []
small_latency = [] 

vosk_us_acc_shadwing = []
vosk_us_latency = [] 

vosk_small_acc_shadwing = []
vosk_small_latency = [] 


for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    
    
    # Test Tiny
    start = time.time()
    raw_tiny = audio_test_denoised_whisper(i, tiny,8)
    dur_tiny = time.time() - start
    tiny_latency.append(dur_tiny)
    tiny_norm = text_preprocessing(raw_tiny)

    # Test Base
    start = time.time()
    raw_base = audio_test_denoised_whisper(i, base,8)
    dur_base = time.time() - start
    base_latency.append(dur_base)
    base_norm = text_preprocessing(raw_base)

    # Test Small
    start = time.time()
    raw_small = audio_test_denoised_whisper(i, small,8)
    dur_small = time.time() - start
    small_latency.append(dur_small)
    small_norm = text_preprocessing(raw_small)

    
    print(f"Target:        {target_norm}")
    print(f"Whisper Tiny:  {tiny_norm}  (Time: {dur_tiny:.3f}s)")
    print(f"Whisper Base:  {base_norm}  (Time: {dur_base:.3f}s)")
    print(f"Whisper Small: {small_norm}  (Time: {dur_small:.3f}s)")
    print("\n\n")
    
    score_tiny, report_tiny = sequance_matching_score(target_norm, tiny_norm)
    score_base, report_base = sequance_matching_score(target_norm, base_norm)
    score_small, report_small = sequance_matching_score(target_norm, small_norm)
    
    tiny_acc_shadwing.append(score_tiny)
    base_acc_shadwing.append(score_base)
    small_acc_shadwing.append(score_small)


    print("--- TINY REPORT ---")
    for item in report_tiny:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- BASE REPORT ---")
    for item in report_base:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    print("--- SMALL REPORT ---")
    for item in report_small:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    
    
    print(f"Scores -> Tiny: {score_tiny} | Base: {score_base} | Small: {score_small} ")
    
    print("\n\n")
    print("__________________________________________________________________")



# Shadowing Pass Rate (%)
threshold = 70
shadow_tiny_pass = [score >= threshold for score in tiny_acc_shadwing]
shadow_base_pass = [score >= threshold for score in base_acc_shadwing]
shadow_small_pass = [score >= threshold for score in small_acc_shadwing]


shadow_tiny_acc = sum(shadow_tiny_pass) / len(shadow_tiny_pass) * 100
shadow_base_acc = sum(shadow_base_pass) / len(shadow_base_pass) * 100
shadow_small_acc = sum(shadow_small_pass) / len(shadow_small_pass) * 100

# Shadowing Average Score
shadow_tiny_avg = sum(tiny_acc_shadwing) / len(tiny_acc_shadwing)
shadow_base_avg = sum(base_acc_shadwing) / len(base_acc_shadwing)
shadow_small_avg = sum(small_acc_shadwing) / len(small_acc_shadwing)

# Average Latency (Time)
avg_time_tiny = sum(tiny_latency) / len(tiny_latency)
avg_time_base = sum(base_latency) / len(base_latency)
avg_time_small = sum(small_latency) / len(small_latency)

print("\n=== FINAL RESULTS ===")

print(f"Shadowing Accuracy  Tiny:  {shadow_tiny_acc:.2f}%")
print(f"Shadowing Accuracy  Base:  {shadow_base_acc:.2f}%")
print(f"Shadowing Accuracy Small: {shadow_small_acc:.2f}%")

print("-" * 30)
print(f"Shadowing Avg Score - Tiny:  {shadow_tiny_avg:.2f}")
print(f"Shadowing Avg Score - Base:  {shadow_base_avg:.2f}")
print(f"Shadowing Avg Score - Small: {shadow_small_avg:.2f}")

print("-" * 30)
print(f"Avg Latency (sec) - Tiny:  {avg_time_tiny:.4f}s")
print(f"Avg Latency (sec) - Base:  {avg_time_base:.4f}s")
print(f"Avg Latency (sec) - Small: {avg_time_small:.4f}s")


Target:        ['august']
Whisper Tiny:  ['oh', 'very', 'good']  (Time: 12.743s)
Whisper Base:  ['oh', 'baby']  (Time: 6.575s)
Whisper Small: ['oh', 'good']  (Time: 34.711s)



--- TINY REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- BASE REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
--- SMALL REPORT ---
august          | Wrong word. Heard 'oh'
_____________________________________
Scores -> Tiny: 0 | Base: 0 | Small: 0 



__________________________________________________________________
Target:        ['may']
Whisper Tiny:  ['me']  (Time: 7.552s)
Whisper Base:  ['may']  (Time: 3.367s)
Whisper Small: ['me']  (Time: 9.789s)



--- TINY REPORT ---
may             | Wrong word. Heard 'me'
_____________________________________
--- BASE REPORT ---
may             | match
_____________________________________
--- SMALL REPORT ---
may             | Wrong word. Heard 'me'
_______________________________

| Model | Accuracy | Avg Accuracy | Latency | VAD Accuracy | Avg VAD Accuracy | VAD Latency | Beam 8 Accuracy | Avg Beam Accuracy | Beam 8 Latency | Model Size |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **US Model** | 62.91% | 73.24% | 1.3543s | --- | --- | --- | --- | --- | --- | 2.9 GB |
| **Small Model** | 58.94% | 69.64% | 0.2151s | --- | --- | --- | --- | --- | --- | 70.9 MB |
| **Whisper Tiny** | 67.55% | 77.54% | 0.7921s | 68.87% | 78.98% | 0.9190s | 68.87% | 78.79% | 1.0724s | 80 MB |
| **Whisper Base** | 84.11% | 87.52% | 1.3941s | 84.11% | 88.02% | 1.5631s | 84.11% | 87.69% | 1.7275s | 150 MB|
| **Whisper Small** | **87.42%** | **90.07%** | 3.9090s | **86.75%** | **89.25%** | 3.9992s | **87.42%** | **89.72%** | 4.6112s | 490 MB |

In [ ]:
=== FINAL RESULTS ===
Shadowing Accuracy  Tiny:  67.55%
Shadowing Accuracy  Base:  84.11%
Shadowing Accuracy Small: 87.42%
Shadowing Accuracy Vosk US:    62.91%
Shadowing Accuracy Vosk Small: 58.94%
------------------------------
Shadowing Avg Score - Tiny:  77.93
Shadowing Avg Score - Base:  87.52
Shadowing Avg Score - Small: 90.07
Shadowing Avg Score - Vosk US:    73.24
Shadowing Avg Score - Vosk Small: 69.64
------------------------------
Avg Latency (sec) - Tiny:  0.7921s
Avg Latency (sec) - Base:  1.3941s
Avg Latency (sec) - Small: 3.9090s
Avg Latency (sec) - Vosk US:    1.3543s
Avg Latency (sec) - Vosk Small: 0.2151s